# Simulation with large scale forcing from ERA5
This notebook showcases the use of large scale forcing to drive MicroHH simulations. It is based on a simplified version of the `cabauw` case in [`L2SD`](https://github.com/LS2D/LS2D/blob/main/examples/microhh/cabauw_input.py). The full example can be found in `/microhh/cases/cabauw/`

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
import os
import pyproj
import dask

import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import numpy as np
import pandas as pd
import seaborn as sns
import xarray as xr
from datetime import datetime
from pathlib import Path


import microhh_processing_chain.cscs as cscs
import microhh_processing_chain.enums as enums
import microhh_processing_chain.preprocessing as pre
import microhh_processing_chain.postprocessing as post
import microhh_processing_chain.visualisation as vis
import microhh_processing_chain.initialisation as init
import microhh_processing_chain.wind as wind_proc
import microhh_processing_chain.misc as misc
import microhh_processing_chain.io as io

import ls2d
import ucat
from microhh_py import microhh_tools as mht


In [ ]:
# set logging level to info
import logging

logging.basicConfig(level=logging.INFO)

In [ ]:
# get the username of the current user
import getpass

user = getpass.getuser()
print(f"Current user: {user}")

## Initialize case

In [ ]:
scratch_path = Path(f"/capstor/scratch/cscs/{user}")
store_path = Path(f"/capstor/store/cscs/empa/em05/{user}")
era5_data_path = scratch_path / "era5_data"
era5_data_path.mkdir(parents=True, exist_ok=True)

In [ ]:
case_suffix = "first_run"

case = init.initialise_case(
    name="large_scale_forcing",
    suffix=case_suffix,
    scratch_path=scratch_path,
    method=enums.InitMethod.INTERACTIVE,
)

In [ ]:
# set grid:
itot = jtot = ktot = 128

dx = dy = 1
dz = dx / 2

In [ ]:
# set coordinates of Zurich
lat = 47.3786
lon = 8.5400

In [ ]:
# Define vertical grid LES:
grid = ls2d.grid.Grid_equidist(kmax=ktot, dz0=dz)
grid.plot()

In [ ]:
# Constants
Rd = 287.04
Rv = 461.5
ep = Rd / Rv

The following code cell submitts requests to the Climate Data Store (CDS) for the required ERA5 data. As these requests can take some time to process, the code is designed to exit after submitting the requests, allowing you to check the CDS website for the status of your requests. The metadata of the submitted requests is saved in a `.pickle` file which is used by `ls2d` to check the status of the requests and download the data once it's ready. If your request finished successfully, you can re-run the cell to download the data and proceed with the processing chain. The data will be downloaded to the specified `era5_data_path`. The code checks if the data already exists before making a request, so you can safely re-run this cell without worrying about duplicate downloads.

In [ ]:
# Download ERA5 and generate LES initialisation and forcings

settings = {
    "central_lat": lat,
    "central_lon": lon,
    "area_size": 1,
    "case_name": "zuerich",
    "era5_path": str(era5_data_path),
    "era5_expver": 1,  # 1=normal ERA5, 5=ERA5 near-realtime
    "start_date": pd.Timestamp("2024-09-16 00:00"),
    "end_date": pd.Timestamp("2024-09-16 23:00"),
    "write_log": True,
    "data_source": "CDS",
}

# Download required ERA5 files:
ls2d.download_era5(settings)

<div class="alert alert-block alert-warning">
Do not continue with the rest of the notebook until your CDS requests have been completed and the data has been downloaded. You can check the status of your requests on the CDS website.
</div>

In [ ]:
# Read ERA5 data, and calculate derived properties (thl, etc.):
era = ls2d.Read_era5(settings)

# Calculate initial conditions and large-scale forcings for LES:
era.calculate_forcings(n_av=0, method="4th")

# Interpolate ERA5 variables and forcings onto LES grid.
# In addition, `get_les_input` returns additional variables needed to init LES.
les_input = era.get_les_input(grid.z)

In [ ]:
# Remove top ERA5 level, to ensure that pressure stays
# above the minimum reference pressure in RRTMGP
les_input = les_input.sel(lay=slice(0, 135), lev=slice(0, 136))

#
# MicroHH specific initialisation
#
# Settings:
# ------------------
# Root fraction coefficients (see IFS documentation):
a_r = 10.739
b_r = 2.608

# Nudging time scale atmosphere
tau_nudge = 10800
# ------------------

# Fixed background concentrations RRTMGP:
co2 = 348.0e-6
ch4 = 1650.0e-9
n2o = 306.0e-9
n2 = 0.7808
o2 = 0.2095

# Mean radiation profiles on LES grid:
qt_les = les_input["qt"].mean(axis=0)
o3_les = les_input["o3"].mean(axis=0)

# Conversion moisture from mass to volume mixing ratio
h2o_les = qt_les / (ep - ep * qt_les)

# Nudge factor
nudge_fac = np.ones(grid.kmax) / tau_nudge

# Write NetCDF input file for MicroHH
init_profiles = {
    "z": grid.z,
    "thl": les_input["thl"][0, :],
    "qt": les_input["qt"][0, :],
    "u": les_input["u"][0, :],
    "v": les_input["v"][0, :],
    "nudgefac": nudge_fac,
    "co2": co2,
    "ch4": ch4,
    "n2o": n2o,
    "n2": n2,
    "o2": o2,
    "o3": o3_les * 1e-6,
    "h2o": h2o_les,
}

timedep_surface = {
    "time_surface": les_input["time_sec"],
    "p_sbot": les_input["ps"],
    "thl_sbot": les_input["wth"],
    "qt_sbot": les_input["wq"],
}

timedep_ls = {
    "time_ls": les_input["time_sec"],
    "u_geo": les_input["ug"],
    "v_geo": les_input["vg"],
    "w_ls": les_input["wls"],
    "thl_ls": les_input["dtthl_advec"],
    "qt_ls": les_input["dtqt_advec"],
    "u_ls": les_input["dtu_advec"],
    "v_ls": les_input["dtv_advec"],
    "thl_nudge": les_input["thl"],
    "qt_nudge": les_input["qt"],
    "u_nudge": les_input["u"],
    "v_nudge": les_input["v"],
}


io.save_case_input(
    case.name,
    case.scratch_path,
    init_profiles=init_profiles,
    tdep_surface=timedep_surface,
    tdep_ls=timedep_ls,
)

In [ ]:
ls_vis = vis.CaseViewer(case)
figs = ls_vis.plot_initial_conditions()

Set variables in the `.ini` file

In [ ]:
ic = io.open_initial_cond(case)

In [ ]:
# source specific settings
source_x0 = 20
source_y0 = 64
source_z0 = 10

strength = ucat.convert_mass_per_time_unit(100, "kg/h", "kg/s")  # in kg/s
sigma_x = dx / 4
sigma_y = dy / 4
sigma_z = dz / 4

In [ ]:
settings = {
    "master": {
        "npx": 16,
        "npy": 8,
    },
    "grid": {
        "itot": itot,
        "jtot": jtot,
        "ktot": ktot,
        "xsize": itot * dx,
        "ysize": jtot * dy,
        "zsize": grid.zsize,
        "lat": lat,
        "lon": lon,
    },
    "thermo": {
        "pbot": np.nanmean(ic.p_sbot),
        "swtimedep_pbot": False,
    },
    "land_surface": {
        "sw_land_surface": True,
        "ktot": 4,
        "sw_homogeneous": True,
        "sw_free_drainage": True,
        "gD": 0.0,
        "c_veg": 0.95,
        "lai": 2.6,
        "rs_veg_min": 100,
        "rs_soil_min": 50,
        "lambda_stable": 10,
        "lambda_unstable": 10,
        "cs_veg": 0,
    },
    "force": {
        "swlspres": "geo",
        "swtimedep_geo": True,
        "fc": les_input.attrs["fc"],
        "swls": True,
        "swtimedep_ls": True,
        "lslist": ["thl", "qt", "u", "v"],
        "timedeplist_ls": ["thl", "qt", "u", "v"],
        "swnudge": True,
        "swtimedep_nudge": True,
        "nudgelist": ["thl", "qt", "u", "v"],
        "timedeplist_nudge": ["thl", "qt", "u", "v"],
        "swwls": True,
        "swtimedep_wls": True,
    },
    "boundary": {
        "mbcbot": "noslip",
        "mbctop": "freeslip",
        "sbcbot": "dirichlet",
        "sbctop": "neumann",
        "sbot": 0,
        "stop": 0,
        "sbot[thl]": None,
        "stop[thl]": None,
        "sbot[qt]": None,
        "stop[qt]": None,
        "swboundary": "surface",
        "sbcbot": "flux",
        "swtimedep": True,
        "timedeplist": ["thl_sbot", "qt_sbot"],
        "scalar_outflow": "ch4",
        "flow_direction[north]": "outflow",
        "flow_direction[east]": "outflow",
        "flow_direction[south]": "outflow",
        "flow_direction[west]": "outflow",
    },
    "fields": {
        "visc": 0,
        "svisc": 0,
        "rndamp[thl]": 0.1,
        "rndamp[qt]": 0.0001,
        "rndz": grid.zsize * 0.5,
        "rndexp": 2.0,
    },
    "buffer": {
        "swbuffer": True,
        "swupdate": True,
        "zstart": grid.zsize * 3 / 4.0,
        "sigma": 0.00223,
        "beta": 2.0,
    },
    "source": {
        "sourcelist": ["ch4"],
        "source_x0": source_x0,
        "source_y0": source_y0,
        "source_z0": source_z0,
        "sigma_x": sigma_x,
        "sigma_y": sigma_y,
        "sigma_z": sigma_z,
        "strength": strength,
        "swvmr": False,
        "swtimedep_strength": False,
    },
    "time": {
        "savetime": 60,
        "endtime": float(les_input["time_sec"][-1]),
    },
}

case.overwrite_ini(settings)

In [ ]:
# case.open_mhh_in_vscode("ini")

## Pre-process case

### Create bash script and run checks to validate the setup

In [ ]:
# DEFINE ALPS SETUP
alps_setup = cscs.AlpsSetup(
    cluster="eiger",
    account="em05",
    processor="cpu",
    run_time="00:30:00",
    partition="debug",
    ntasks_per_core=1,
    ntasks_per_node=128,
    uenv="prgenv-gnu/24.11:v2",
    view="default",
)

# DO PRE-PROCESSING
pre_processing = pre.PreProcessing(
    case,
    alps_setup,
    microhh_executable_path=f"/users/{user}/microhh/build.mpi",
)
pre_processing.prepare_simulation(overwrite=True)